# Run All Experiments (Notebook)

**Workflow:**
1. Run "Imports" and "Configuration" cells once
2. Run "Load Data" cell once (slow, tokenization)
3. Edit strategy/experiment config and re-run "Run Experiments" cell as needed — data stays in memory, strategies are reloaded from disk each time

In [1]:
## Imports (run once)

from __future__ import annotations

import importlib
import multiprocessing as mp
import sys
import time
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

_ROOT = Path(".").resolve().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.config import DEFAULT_TOKENIZER_NAME, RESULTS_DIR
from src.model_config import DEFAULT_MODEL, ModelConfig
from experiments.runner import (
    capacity_from_spec,
    effective_page_size,
    persist_result_row,
    prepare_requests,
    run_multi_tier_simulation,
    run_simulation,
)
from viz.tree_logger import TreeLogger, _log_filename

print(f"Project root: {_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Default model: {DEFAULT_MODEL.name}  (available: {', '.join(ModelConfig.list_models())})")

Project root: /data/howarli/dev/LLM-prefix-caching-simulator
Results dir:  /data/howarli/dev/LLM-prefix-caching-simulator/results
Default model: qwen3.5-27b  (available: jamba-1.5-mini, llama-3.1-70b, llama-3.1-8b, qwen3-0.6b, qwen3.5-27b, transformer-28l-4096d)


## Configuration

Edit these and re-run. The "Load Data" cell uses `DATASETS`, `ORDERING`, `TOKENIZER`, `SEED`, `MAX_REQUESTS`, `TOKENIZE_WORKERS`.

In [2]:
# ── Data loading config ─────────────────────────────────────────────
DATASETS        = ["swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]
ORDERING        = "timestamp"
TOKENIZER       = DEFAULT_TOKENIZER_NAME
SEED            = 0
MAX_REQUESTS    = 5000
TOKENIZE_WORKERS = 90

# ── Model config ───────────────────────────────────────────────────
MODEL_NAME      = DEFAULT_MODEL.name  # see ModelConfig.list_models()

# ── Multi-tier (HBM + DRAM) cache ──────────────────────────────────
# Set MULTI_TIER=True to run two-tier simulations. When enabled, the single-tier
# capacity/strategy fields per-experiment are ignored; HBM_*/DRAM_* take over.
# Per-experiment overrides via "multi_tier", "hbm_capacity", "dram_capacity",
# "hbm_strategy", "dram_strategy".
MULTI_TIER      = False
HBM_CAPACITY    = "20"           # GB (must be finite)
DRAM_CAPACITY   = "160"          # GB (must be finite)
HBM_STRATEGY    = "branch"       # eviction policy for HBM tier
DRAM_STRATEGY   = "marconi3_ev1_mn0"  # eviction policy for DRAM tier

# ── Logging ─────────────────────────────────────────────────────────
ENABLE_LOG      = False
LOG_DIR         = "viz/logs"

# ── Parallelism ─────────────────────────────────────────────────────
MAX_WORKERS     = 100

## Load Data (run once)

Tokenizes all datasets and caches the results in `PREPARED`. Re-run this cell only if you change `DATASETS`, `ORDERING`, `MAX_REQUESTS`, etc.

In [3]:
# PREPARED: dict mapping (dataset, ordering, tokenizer, max_requests, seed) -> List[TokenizedRequest]
PREPARED: Dict[tuple, list] = {}

for ds in DATASETS:
    prep_key = (ds, ORDERING, TOKENIZER, MAX_REQUESTS, SEED)
    if prep_key in PREPARED:
        print(f"  [skip] {ds}/{ORDERING} already loaded ({len(PREPARED[prep_key])} requests)")
        continue
    t0 = time.perf_counter()
    print(f"  Preparing: {ds}/{ORDERING} (max_requests={MAX_REQUESTS}) ...", end=" ", flush=True)
    reqs = prepare_requests(
        ds, ORDERING, TOKENIZER,
        seed=SEED,
        tokenize_workers=TOKENIZE_WORKERS,
        max_requests=MAX_REQUESTS,
    )
    elapsed = time.perf_counter() - t0
    PREPARED[prep_key] = reqs
    print(f"{len(reqs)} requests ready ({elapsed:.1f}s)")

print(f"\nTotal prep keys loaded: {len(PREPARED)}")

  Preparing: swesmith/timestamp (max_requests=5000) ... 

Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/47 [00:00<?, ?it/s]

5000 requests ready (51.3s)
  Preparing: loogle/timestamp (max_requests=5000) ... 1951 requests ready (8.2s)
  Preparing: narrativeqa/timestamp (max_requests=5000) ... 

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

1494 requests ready (26.1s)
  Preparing: sharegpt_90k_raw/timestamp (max_requests=5000) ... 5000 requests ready (17.8s)

Total prep keys loaded: 4


## Run Experiments

Edit the `EXPERIMENTS` list below and re-run this cell. Strategy modules are **reloaded from disk** each time so you can modify strategy code without restarting the kernel or reloading data.

In [7]:
# ── Reload all modules from disk ────────────────────────────────────
# This picks up any code changes without restarting the kernel.
import src.model_config
import src.radix_tree
import src.cache_simulator
import src.metrics
import src.strategies.base
import src.strategies.lru
import src.strategies.lfu
import src.strategies.fifo
import src.strategies.marconi
import src.strategies.marconi2
import src.strategies.marconi3
import src.strategies.branch
import src.strategies.crf_decoupling
import src.strategies
import experiments.runner

for mod in [
    src.model_config,
    src.radix_tree,
    src.cache_simulator,
    src.metrics,
    src.strategies.base,
    src.strategies.lru,
    src.strategies.lfu,
    src.strategies.fifo,
    src.strategies.marconi,
    src.strategies.marconi2,
    src.strategies.marconi3,
    src.strategies.crf_decoupling,
    src.strategies,
    experiments.runner,
]:
    importlib.reload(mod)

from experiments.runner import (
    capacity_from_spec,
    effective_page_size,
    persist_result_row,
    run_multi_tier_simulation,
    run_simulation,
    strategy_from_name,
)
from src.model_config import ModelConfig
print("All modules reloaded.")

All modules reloaded.


In [ ]:
# ── Define experiments ──────────────────────────────────────────────
# Edit this list freely and re-run from the reload cell above.

EXPERIMENTS: list[dict] = []

for ds in ["swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]:
# for ds in ["sharegpt_90k_raw"]:
    # for page_size in [1, 32, 256, 1024]:
    for page_size in [32]:
        for capacity in [10, 20, 40, 80, 120, 160, "inf"]:
        # for capacity in [10, 20]:
            pass
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi2", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3", capacity=capacity))
            EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev0_nt", capacity=capacity))
            EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev1_nt", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev2_mn0", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev3_mn0", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev3_mn1", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="branch", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="branch_nt", capacity=capacity))


for ds in ["swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]:
    for page_size in [32]:
        for capacity in [10, 20]:
            EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, multi_tier=True, hbm_capacity=capacity, dram_capacity="inf", hbm_strategy="branch_nt", dram_strategy="marconi3_ev1_nt"))

print(f"Total experiments: {len(EXPERIMENTS)}")

Total experiments: 56


In [9]:
# ── Run all experiments ─────────────────────────────────────────────

def _build_strategy_name(cfg: dict) -> str:
    name = cfg["strategy"]
    n = name.lower()
    if n in ("marconi", "marconi2") and cfg.get("marconi_alpha") is not None:
        return f"{name}_{cfg['marconi_alpha']}"
    if n == "crf_decoupling" and cfg.get("crf_lambda_decay") is not None:
        return f"{name}_{cfg['crf_lambda_decay']}"
    return name

def _merge_defaults(cfg: dict) -> dict:
    return {
        "dataset":                cfg.get("dataset", DATASETS[0]),
        "page_size":              cfg.get("page_size", 32),
        "ordering":               cfg.get("ordering", ORDERING),
        "strategy":               cfg.get("strategy", "lru"),
        "capacity":               cfg.get("capacity", "160"),
        "model_name":             cfg.get("model_name", MODEL_NAME),
        "tokenizer":              cfg.get("tokenizer", TOKENIZER),
        "seed":                   cfg.get("seed", SEED),
        "max_requests":           cfg.get("max_requests", MAX_REQUESTS),
        "marconi_alpha":          cfg.get("marconi_alpha"),
        "crf_lambda_decay":       cfg.get("crf_lambda_decay"),
        "crf_c_attn":             cfg.get("crf_c_attn"),
        "crf_c_ssm":              cfg.get("crf_c_ssm"),
        # Multi-tier params
        "multi_tier":             cfg.get("multi_tier", MULTI_TIER),
        "hbm_capacity":           cfg.get("hbm_capacity", HBM_CAPACITY),
        "dram_capacity":          cfg.get("dram_capacity", DRAM_CAPACITY),
        "hbm_strategy":           cfg.get("hbm_strategy", HBM_STRATEGY),
        "dram_strategy":          cfg.get("dram_strategy", DRAM_STRATEGY),
    }

def _label(cfg: dict) -> str:
    if cfg.get("multi_tier"):
        return (
            f"{cfg['dataset']} ps={cfg['page_size']} {cfg['ordering']} "
            f"hbm:{cfg['hbm_strategy']}/{cfg['hbm_capacity']} "
            f"dram:{cfg['dram_strategy']}/{cfg['dram_capacity']}"
        )
    return f"{cfg['dataset']} ps={cfg['page_size']} {cfg['ordering']} {cfg['strategy']} cap={cfg['capacity']}"

def _sim_worker(args):
    """Run one simulation inside a pool worker (fork inherits PREPARED)."""
    prep_key, page_size, sim_spec, cfg = args
    reqs = PREPARED[prep_key]
    model = ModelConfig.from_name(cfg["model_name"])

    if cfg.get("multi_tier"):
        hbm_strat = strategy_from_name(sim_spec["hbm_strategy"], model=model)
        dram_strat = strategy_from_name(sim_spec["dram_strategy"], model=model)
        # NOTE: TreeLogger is not currently wired through run_multi_tier_simulation.
        metrics = run_multi_tier_simulation(
            reqs, page_size, hbm_strat, dram_strat,
            hbm_capacity_tokens=sim_spec["hbm_cap"],
            dram_capacity_tokens=sim_spec["dram_cap"],
            model=model,
        )
        return {"cfg": cfg, "metrics": metrics.to_dict()}

    strategy_name = sim_spec["strategy_name"]
    cap = sim_spec["cap"]
    strategy = strategy_from_name(strategy_name, model=model)

    logger = None
    if ENABLE_LOG:
        log_dir = Path(LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        fname = _log_filename(
            dataset=cfg["dataset"], strategy=strategy_name,
            page_size=page_size, capacity_spec=str(cfg["capacity"]),
            ordering=cfg["ordering"], mamba_equiv=model.mamba_state_token_equiv,
        )
        logger = TreeLogger(log_dir / fname)

    try:
        metrics = run_simulation(reqs, page_size, strategy, cap, logger=logger, model=model)
    finally:
        if logger is not None:
            logger.close()

    return {"cfg": cfg, "metrics": metrics.to_dict()}


# Build simulation args
merged = [_merge_defaults(cfg) for cfg in EXPERIMENTS]
sim_args = []
for cfg in merged:
    ds = cfg["dataset"]
    model = ModelConfig.from_name(cfg["model_name"])
    prep_key = (ds, cfg["ordering"], cfg["tokenizer"], cfg["max_requests"], cfg["seed"])
    if prep_key not in PREPARED:
        print(f"  [WARNING] Data not loaded for {prep_key}, skipping")
        continue
    page_size = effective_page_size(ds, int(cfg["page_size"]))
    if cfg.get("multi_tier"):
        hbm_cap = capacity_from_spec(str(cfg["hbm_capacity"]), model=model)
        dram_cap = capacity_from_spec(str(cfg["dram_capacity"]), model=model)
        if hbm_cap is None or dram_cap is None:
            raise SystemExit(
                f"multi_tier requires finite hbm_capacity/dram_capacity (got "
                f"hbm={cfg['hbm_capacity']!r}, dram={cfg['dram_capacity']!r})"
            )
        sim_spec = {
            "hbm_strategy": cfg["hbm_strategy"],
            "dram_strategy": cfg["dram_strategy"],
            "hbm_cap": hbm_cap,
            "dram_cap": dram_cap,
        }
    else:
        cap = capacity_from_spec(str(cfg["capacity"]), model=model)
        sim_spec = {
            "strategy_name": _build_strategy_name(cfg),
            "cap": cap,
        }
    sim_args.append((prep_key, page_size, sim_spec, cfg))

print(f"Running {len(sim_args)} simulation(s) (model={MODEL_NAME}, max_workers={MAX_WORKERS}) ...")
t_start = time.perf_counter()

ctx = mp.get_context("fork")
done = 0

with ctx.Pool(processes=MAX_WORKERS) as pool:
    for result in pool.imap_unordered(_sim_worker, sim_args):
        cfg = result["cfg"]
        metrics_dict = result["metrics"]

        dataset = cfg["dataset"]
        out_csv = RESULTS_DIR / f"results_{dataset}.csv"
        out_json_dir = RESULTS_DIR / f"json_{dataset}"

        model = ModelConfig.from_name(cfg["model_name"])
        if cfg.get("multi_tier"):
            strategy_label = f"hbm:{cfg['hbm_strategy']}_dram:{cfg['dram_strategy']}"
            capacity_label = f"hbm{cfg['hbm_capacity']}_dram{cfg['dram_capacity']}"
        else:
            strategy_label = _build_strategy_name(cfg)
            capacity_label = str(cfg["capacity"])
        row = {
            "dataset": dataset,
            "page_size": effective_page_size(dataset, int(cfg["page_size"])),
            "ordering": cfg["ordering"],
            "strategy": strategy_label,
            "capacity_spec": capacity_label,
            "tokenizer": cfg["tokenizer"],
            "mamba_state_token_equiv": model.mamba_state_token_equiv,
            "model_name": cfg["model_name"],
            "metrics": metrics_dict,
        }
        persist_result_row(out_csv, out_json_dir, row)

        done += 1
        token_hr = metrics_dict.get("token_level_hit_rate", 0)
        flops_sr = metrics_dict.get("flops_save_rate")
        flops_str = f"  flops_save={flops_sr:.4f}" if flops_sr is not None else ""
        if cfg.get("multi_tier"):
            hbm_hr = metrics_dict.get("hbm_token_hit_rate", 0)
            dram_hr = metrics_dict.get("dram_token_hit_rate", 0)
            tier_str = f"  hbm_hr={hbm_hr:.4f}  dram_hr={dram_hr:.4f}"
        else:
            tier_str = ""
        print(f"  [{done}/{len(sim_args)}] {_label(cfg)}  token_hr={token_hr:.4f}{tier_str}{flops_str}", flush=True)

elapsed = time.perf_counter() - t_start
print(f"\nAll {len(sim_args)} experiments completed in {elapsed:.1f}s.")

Running 56 simulation(s) (model=qwen3.5-27b, max_workers=100) ...
  [1/56] loogle ps=32 timestamp marconi3_ev1_nt cap=inf  token_hr=0.9049  flops_save=0.9057
  [2/56] loogle ps=32 timestamp marconi3_ev0_nt cap=10  token_hr=0.4077  flops_save=0.4059
  [3/56] loogle ps=32 timestamp marconi3_ev1_nt cap=20  token_hr=0.6100  flops_save=0.6083
  [4/56] sharegpt_90k_raw ps=32 timestamp marconi3_ev1_nt cap=10  token_hr=0.7435  flops_save=0.7567
  [5/56] loogle ps=32 timestamp marconi3_ev0_nt cap=20  token_hr=0.5788  flops_save=0.5772
  [6/56] loogle ps=32 timestamp marconi3_ev1_nt cap=10  token_hr=0.4179  flops_save=0.4162
  [7/56] loogle ps=32 timestamp marconi3_ev0_nt cap=inf  token_hr=0.9049  flops_save=0.9057
  [8/56] swesmith ps=32 timestamp marconi3_ev0_nt cap=inf  token_hr=0.9708  flops_save=0.9715
  [9/56] swesmith ps=32 timestamp marconi3_ev1_nt cap=10  token_hr=0.4330  flops_save=0.4368
  [10/56] loogle ps=32 timestamp marconi3_ev1_nt cap=40  token_hr=0.8728  flops_save=0.8731
  [11/